Read silver table

In [0]:
from pyspark.sql import functions as F

employee_silver_df = spark.table(
    "databricks_project1.silver.employee_payroll"
)

print("Silver Employee table loaded successfully")
print(f"Silver Employee row count: {employee_silver_df.count()}")

employee_silver_df.printSchema()

Select and standardize employee _data_

In [0]:
employee_base_df = (
    employee_silver_df
    .select(
        F.col("Employee_Code").cast("string").alias("Employee_Code"),
        F.col("Badge_#").cast("int").alias("Badge_#"),
        F.trim(F.col("Employee_Status")).alias("Employee_Status"),
        F.trim(F.col("FirstName")).alias("First_Name"),
        F.trim(F.col("LastName")).alias("Last_Name"),
        F.col("Facility_Code").cast("int").alias("Facility_Code"),
        F.trim(F.col("facname")).alias("Facility_Name"),
        F.trim(F.col("Incharge")).alias("Incharge"),
        F.col("Labor_Position_Code").cast("int").alias("Labor_Position_Code"),
        F.col("DOB").alias("Birth_Date"),
        F.col("Employee_Added").alias("Employee_Added"),
        F.col("Hire_Date").alias("Hire_Date"),
        F.col("Rehire_Date").alias("Rehire_Date"),
        F.col("Termination_Date").alias("Termination_Date"),
        F.col("ingestion_timestamp").alias("ingestion_timestamp"),
        F.col("source_file").alias("source_file")
    )
)

print(f"Base employee row count: {employee_base_df.count()}")

display(employee_base_df.limit(10))

Check duplicate business keys

In [0]:
duplicate_employee_df = (
    employee_base_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    f"Duplicate Employee_Code count: "
    f"{duplicate_employee_df.count()}"
)

display(duplicate_employee_df)

Create the SCD Type 2 Gold DataFrame

In [0]:
from pyspark.sql.window import Window

current_date = F.current_date()

final_employee_df = (
    employee_base_df
    .withColumn("EmployeeKey", F.monotonically_increasing_id())
    .withColumn("StartDate", current_date)
    .withColumn(
        "EndDate",
        F.to_date(F.lit("9999-12-31"))
    )
    .withColumn("IsCurrent", F.lit(True))
    .select(
        "EmployeeKey",
        "Employee_Code",
        "Badge_#",
        "Employee_Status",
        "First_Name",
        "Last_Name",
        "Facility_Code",
        "Facility_Name",
        "Incharge",
        "Labor_Position_Code",
        "Birth_Date",
        "Employee_Added",
        "Hire_Date",
        "Rehire_Date",
        "Termination_Date",
        "StartDate",
        "EndDate",
        "IsCurrent",
        "ingestion_timestamp",
        "source_file"
    )
)

print(f"Final DimEmployee count: {final_employee_df.count()}")

final_employee_df.printSchema()
display(final_employee_df.limit(10))

create the gold table

In [0]:
(
    final_employee_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "databricks_project1.gold.dim_employee"
    )
)

verify the table

In [0]:
gold_employee_df = spark.table(
    "databricks_project1.gold.dim_employee"
)

print(f"Gold DimEmployee count: {gold_employee_df.count()}")

display(gold_employee_df)

SCD Validation

In [0]:
current_record_count = (
    gold_employee_df
    .filter(F.col("IsCurrent") == True)
    .count()
)

print(f"Current employee records: {current_record_count}")

invalid_scd_df = gold_employee_df.filter(
    (F.col("StartDate").isNull()) |
    (F.col("EndDate").isNull()) |
    (F.col("IsCurrent").isNull())
)

print(
    f"Invalid SCD records: {invalid_scd_df.count()}"
)

In [0]:
invalid_scd_df = gold_employee_df.filter(
    (F.col("StartDate").isNull()) |
    (F.col("EndDate").isNull()) |
    (F.col("IsCurrent").isNull())
)

print(
    f"Invalid SCD records: {invalid_scd_df.count()}"
)

Audit-column validation

In [0]:
audit_null_count = gold_employee_df.filter(
    F.col("ingestion_timestamp").isNull() |
    F.col("source_file").isNull()
).count()

print(f"Rows with NULL audit columns: {audit_null_count}")

In [0]:
gold_employee_df.groupBy("IsCurrent").count().show()

In [0]:
gold_employee_df.printSchema()

In [0]:
#Temp cell

# ---------------------------------------------------------
# Inspect current DimEmployee state
# ---------------------------------------------------------
from pyspark.sql import functions as F
dim_employee_check_df = spark.table(
    "databricks_project1.gold.dim_employee"
)

print(
    "Total DimEmployee rows:",
    dim_employee_check_df.count()
)

print(
    "Distinct Employee_Code:",
    dim_employee_check_df
    .select("Employee_Code")
    .distinct()
    .count()
)

print(
    "Current IsCurrent=True rows:",
    dim_employee_check_df
    .filter(F.col("IsCurrent") == True)
    .count()
)

duplicate_current_df = (
    dim_employee_check_df
    .filter(F.col("IsCurrent") == True)
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Employee_Code with multiple current records:",
    duplicate_current_df.count()
)

display(duplicate_current_df)

In [0]:
#Temp cell

# ---------------------------------------------------------
# Inspect current DimEmployee state
# ---------------------------------------------------------

dim_employee_check_df = spark.table(
    "databricks_project1.gold.dim_employee"
)

print(
    "Total DimEmployee rows:",
    dim_employee_check_df.count()
)

print(
    "Distinct Employee_Code:",
    dim_employee_check_df
    .select("Employee_Code")
    .distinct()
    .count()
)

print(
    "Current IsCurrent=True rows:",
    dim_employee_check_df
    .filter(F.col("IsCurrent") == True)
    .count()
)

duplicate_current_df = (
    dim_employee_check_df
    .filter(F.col("IsCurrent") == True)
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Employee_Code with multiple current records:",
    duplicate_current_df.count()
)

display(duplicate_current_df)